# J9b — Base détaillée des restaurants en Île-de-France

Objectifs :

- extraire les restaurants de la BPE 2025 ;
- préserver les informations de la source ;
- contrôler les identifiants et les doublons ;
- normaliser les catégories et les codes communaux ;
- examiner les adresses et les coordonnées ;
- conserver les indicateurs de qualité géographique ;
- comparer les comptages avec le profil J9 ;
- préparer les analyses locales du J13b et du J14.

Périmètre :
Équipements BPE de type A504.

La présence dans la BPE 2025 ne constitue pas une vérification
de l'ouverture actuelle du restaurant.

In [1]:
#Importer des librairies

import os

os.environ["OMP_NUM_THREADS"] = "1"
os.environ["NUMEXPR_NUM_THREADS"] = "1"
os.environ["MKL_NUM_THREADS"] = "1"

from pathlib import Path
from datetime import datetime, timezone
import csv
import hashlib
import json
import re
import unicodedata

import numpy as np
import pandas as pd
from IPython.display import display

pd.set_option("display.max_columns", 100)
pd.set_option("display.max_rows", 60)
pd.set_option("display.max_colwidth", 100)

print("Pandas :", pd.__version__)
print("NumPy :", np.__version__)

Pandas : 2.2.2
NumPy : 1.26.4


In [3]:
#Définir les restaurants

RACINE = Path.cwd()

if not (RACINE / "data").is_dir():
    if (RACINE.parent / "data").is_dir():
        RACINE = RACINE.parent
    else:
        raise FileNotFoundError(
            "Ouvre le notebook depuis GeoMarketing_IDF "
            "ou son dossier notebooks."
        )

DOSSIER_BPE = RACINE / "/Users/almou/OneDrive/GeoMarketing_IDF/data" / "raw" / "bpe"
DOSSIER_INTERIM = RACINE / "/Users/almou/OneDrive/GeoMarketing_IDF/data" / "interim" / "j9b"
DOSSIER_PROCESSED = RACINE / "/Users/almou/OneDrive/GeoMarketing_IDF/data" / "processed"

DOSSIER_INTERIM.mkdir(parents=True, exist_ok=True)
DOSSIER_PROCESSED.mkdir(parents=True, exist_ok=True)

# À adapter seulement si ton CSV brut porte un autre nom.
FICHIER_BPE = DOSSIER_BPE / "BPE25.csv"

FICHIER_PROFIL_J9 = (
    DOSSIER_PROCESSED / "profil_communes_idf_j9.csv"
)

FICHIER_RESTAURANTS_DETAIL = (
    DOSSIER_PROCESSED / "restaurants_idf_detail_j9b.csv"
)

for fichier in [FICHIER_BPE, FICHIER_PROFIL_J9]:
    if not fichier.is_file():
        raise FileNotFoundError(f"Fichier introuvable : {fichier}")

print("Racine :", RACINE)
print("BPE :", FICHIER_BPE)
print("Profil :", FICHIER_PROFIL_J9)

Racine : C:\Users\almou
BPE : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\raw\bpe\BPE25.csv
Profil : C:\Users\almou\OneDrive\GeoMarketing_IDF\data\processed\profil_communes_idf_j9.csv


In [4]:
#Fonctions de nettoyage

def normaliser_nom_colonne(nom):
    texte = unicodedata.normalize(
        "NFKD",
        str(nom).strip().upper(),
    )

    texte = "".join(
        caractere
        for caractere in texte
        if not unicodedata.combining(caractere)
    )

    return re.sub(
        r"[^A-Z0-9]+",
        "_",
        texte,
    ).strip("_")


def nettoyer_texte(serie):
    resultat = serie.astype("string").str.strip()

    manquants = {
        "",
        "NA",
        "NAN",
        "NONE",
        "NULL",
        "<NA>",
        "_U",
        "_Z",
        "[ND]",
    }

    return resultat.mask(
        resultat.str.upper().isin(manquants)
    )


def normaliser_code_commune(serie):
    resultat = (
        nettoyer_texte(serie)
        .str.replace(r"\.0$", "", regex=True)
    )

    numerique = resultat.str.fullmatch(
        r"\d{1,5}",
        na=False,
    )

    return resultat.where(numerique).str.zfill(5)


def convertir_nombre(serie):
    texte = (
        nettoyer_texte(serie)
        .str.replace("\u00a0", "", regex=False)
        .str.replace("\u202f", "", regex=False)
        .str.replace(" ", "", regex=False)
        .str.replace(",", ".", regex=False)
    )

    return pd.to_numeric(texte, errors="coerce")


def texte_comparable(serie):
    texte = nettoyer_texte(serie).str.upper()

    texte = texte.str.normalize("NFKD")
    texte = texte.str.replace(
        r"[\u0300-\u036f]",
        "",
        regex=True,
    )

    return (
        texte.str.replace(
            r"[^A-Z0-9]+",
            " ",
            regex=True,
        )
        .str.strip()
    )


def enregistrer_csv(table, fichier):
    table.to_csv(
        fichier,
        index=False,
        encoding="utf-8-sig",
    )

    print(f"{fichier.name} : {len(table):,} lignes")

In [5]:
#Charger le profil J9

profil_j9 = pd.read_csv(
    FICHIER_PROFIL_J9,
    dtype={"CODGEO": "string"},
    encoding="utf-8-sig",
)

profil_j9.columns = [
    normaliser_nom_colonne(colonne)
    for colonne in profil_j9.columns
]

if profil_j9.columns.duplicated().any():
    raise ValueError(
        "Le profil contient des noms de colonnes en double."
    )

if "CODGEO" not in profil_j9.columns:
    raise ValueError("CODGEO est absent du profil J9.")

profil_j9["CODGEO"] = normaliser_code_commune(
    profil_j9["CODGEO"]
)

if profil_j9["CODGEO"].isna().any():
    raise ValueError("Des codes communaux du profil sont invalides.")

if not profil_j9["CODGEO"].is_unique:
    raise ValueError("Le profil contient plusieurs lignes par commune.")

codes_profil = set(profil_j9["CODGEO"])

print("Communes du profil :", len(profil_j9))
display(profil_j9.head())

Communes du profil : 1266


,CODGEO,NOM_COMMUNE,DEP,REG,POPULATION_2011,POPULATION_2016,POPULATION_2022,POP_0_14_ANS_2022,POP_15_29_ANS_2022,POP_30_44_ANS_2022,POP_45_59_ANS_2022,POP_60_74_ANS_2022,POP_75_89_ANS_2022,POP_90_ANS_PLUS_2022,POP_MOINS_30_ANS_2022,POP_15_44_ANS_2022,POP_60_ANS_PLUS_2022,POP_75_ANS_PLUS_2022,PART_0_14_ANS_PCT,PART_15_29_ANS_PCT,PART_30_44_ANS_PCT,PART_MOINS_30_ANS_PCT,PART_15_44_ANS_PCT,PART_60_ANS_PLUS_PCT,PART_75_ANS_PLUS_PCT,EVOLUTION_POP_2016_2022,EVOLUTION_POP_2016_2022_PCT,TAUX_ANNUEL_POP_2016_2022_PCT,REVENU_MEDIAN_EUROS,TAUX_PAUVRETE_PCT,EMPLOIS_SALARIES_LIEU_TRAVAIL,NOMBRE_ETABLISSEMENTS,EMPLOIS_SALARIES_POUR_100_HAB,ETABLISSEMENTS_POUR_1000_HAB,NB_RESTAURANTS_TOTAL,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_TYPE_INCONNU,DENSITE_RESTAURANTS_10000_HAB,DENSITE_RESTAURATION_RAPIDE_10000_HAB,DENSITE_RESTAURATION_TRAD_10000_HAB,PART_RESTAURATION_RAPIDE_PCT,PART_RESTAURATION_TRADITIONNELLE_PCT,INDICE_DENSITE_RAPIDE_IDF_BASE100,POP_0_2,POP_3_5,POP_6_10,POP_11_14,POP_15_17,...,PART_EMPLOIS_LT_INDUSTRIE_PCT,PART_EMPLOIS_LT_CONSTRUCTION_PCT,PART_EMPLOIS_LT_COMMERCE_TRANSPORTS_SERVICES_PCT,PART_EMPLOIS_LT_ADMIN_ENSEIGNEMENT_SANTE_SOCIAL_PCT,EMPLOIS_TERTIAIRES_LT,PART_EMPLOIS_TERTIAIRES_LT_PCT,EMPLOIS_15P_LT_POUR_1000_HAB,INDICE_DENSITE_EMPLOIS_IDF_BASE100,NB_RESTAURATION_RAPIDE_1000_EMPLOIS,INDICE_DENSITE_RAPIDE_PAR_EMPLOI_IDF_BASE100,NB_RESTAURANTS_TOTAL_1000_EMPLOIS,NB_LIEUX_TRANSPORT,NB_LIEUX_BUS,NB_STATIONS_METRO,NB_STATIONS_TRAMWAY,NB_GARES_RER,NB_GARES_TRAIN,NB_STATIONS_FUNICULAIRE,NB_LIEUX_FERRES,NB_GARES_STATIONS_TRANSPORT_LOURD,NB_LIEUX_MULTIMODAUX,NB_POLES_MULTIMODAUX_LOURDS,NB_LIGNES_TRANSPORT,NB_MODES_TRANSPORT_PRESENTS,NB_OPERATEURS_TRANSPORT,NB_LIGNES_BUS,NB_LIGNES_METRO,NB_LIGNES_TRAMWAY,NB_LIGNES_RER,NB_LIGNES_TRAIN,NB_LIGNES_FUNICULAIRE,NB_LIGNES_AUTRES,NB_LIGNES_FERREES,NB_LIGNES_TRANSPORT_LOURD,A_BUS,A_METRO,A_TRAMWAY,A_RER,A_TRAIN,A_TRANSPORT_LOURD,NB_LIEUX_TRANSPORT_10000_HAB,NB_LIGNES_TRANSPORT_10000_HAB,NB_LIGNES_FERREES_10000_HAB,NB_GARES_STATIONS_LOURDES_10000_HAB,NB_POLES_MULTIMODAUX_10000_HAB,INDICE_LIEUX_TRANSPORT_IDF_BASE100,INDICE_LIGNES_FERREES_IDF_BASE100,NB_LIGNES_TRANSPORT_1000_EMPLOIS,NB_GARES_STATIONS_LOURDES_1000_EMPLOIS,CLASSE_DESSERTE_TRANSPORT
0,75056,Paris,75,11,2249975.0,2190327.0,2113705.0,274280.561508,513386.756029,457511.658564,387099.215350,302819.494109,154040.332162,24566.982278,787667.317537,970898.414593,481426.808549,178607.314440,12.98,24.29,21.65,37.26,45.93,22.78,8.45,-76622.0,-3.50,-0.59,33650.0,16.8,1.551134e+06,188002.0,73.384590,88.944294,20911,8329,12553,29,0,98.930551,39.404742,59.388609,39.830711,60.030606,200.612151,54155.23178,51427.58154,87388.40827,75287.99165,59522.89891,...,3.653341,3.241342,69.680190,23.374642,1.685340e+06,93.054832,860.799646,177.970034,4.599291,114.005364,11.547097,1298,1181,249,60,12,11,2,305,261,208,182,251,7,33,199,19,4,6,17,1,5,47,42,1,1,1,1,1,1,6.140876,1.187488,0.222358,1.234799,0.861047,40.394200,50.887449,0.138603,0.144125,POLE_MULTIMODAL_LOURD
1,77001,Achères-la-Forêt,77,11,1232.0,1139.0,1183.0,182.135095,159.522993,189.869837,347.331906,204.113784,89.442063,10.584321,341.658088,349.392830,304.140169,100.026384,15.40,13.48,16.05,28.88,29.53,25.71,8.46,44.0,3.86,0.63,34840.0,NaN,1.303068e+02,35.0,11.014944,29.585799,1,0,1,0,0,8.453085,0.000000,8.453085,0.000000,100.000000,0.000000,32.18226,25.01503,65.89736,58.31802,48.54337,...,5.540315,5.927339,74.653710,10.746325,1.701738e+02,85.400035,161.732538,33.438147,0.000000,0.000000,5.191476,4,4,0,0,0,0,0,0,0,0,0,2,1,1,2,0,0,0,0,0,0,0,0,1,0,0,0,0,0,33.812342,16.906171,0.000000,0.000000,0.000000,222.414931,0.000000,10.382952,0.000000,BUS_UNIQUEMENT
2,77002,Amillis,77,11,781.0,819.0,821.0,127.936214,122.351987,146.913593,151.603361,188.399317,59.320602,24.474926,250.288201,269.265581,272.194845,83.795528,15.58,14.90,17.89,30.49,32.80,33.15,10.21,2.0,0.24,0.04,29130.0,NaN,1.906978e+02,39.0,23.227509,47.503045,2,2,0,0,0,24.360536,24.360536,

In [6]:
#Inspecte l'en-tête de BPE

ENCODAGE_BPE = "utf-8-sig"

with open(
    FICHIER_BPE,
    "r",
    encoding=ENCODAGE_BPE,
    newline="",
) as flux:
    premiere_ligne = flux.readline()

try:
    SEPARATEUR_BPE = csv.Sniffer().sniff(
        premiere_ligne,
        delimiters=";,\t|",
    ).delimiter

except csv.Error as erreur:
    raise ValueError(
        "Séparateur non identifié. "
        "Inspecte la première ligne du CSV."
    ) from erreur

entete = pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    encoding=ENCODAGE_BPE,
    nrows=0,
)

renommage_bpe = {
    colonne: normaliser_nom_colonne(colonne)
    for colonne in entete.columns
}

noms_normalises = list(renommage_bpe.values())

if len(noms_normalises) != len(set(noms_normalises)):
    raise ValueError(
        "Des colonnes deviennent identiques après normalisation."
    )

colonnes_requises = {
    "AN",
    "DEPCOM",
    "TYPEQU",
    "TYPERESTO",
}

colonnes_absentes = colonnes_requises - set(noms_normalises)

print("Séparateur :", repr(SEPARATEUR_BPE))
print("Colonnes :", noms_normalises)

if colonnes_absentes:
    raise ValueError(
        f"Colonnes absentes : {sorted(colonnes_absentes)}. "
        "Vérifie que le fichier est bien le détail BPE 2025."
    )

Séparateur : ';'
Colonnes : ['AN', 'APET', 'NOMRS', 'CNOMRS', 'NUMVOIE', 'INDREP', 'TYPVOIE', 'LIBVOIE', 'CADR', 'CODPOS', 'DEPCOM', 'DEP', 'REG', 'LIBCOM', 'DOM', 'SDOM', 'TYPEQU', 'SIRET', 'STATUT_DIFFUSION', 'CANTINE', 'INTERNAT', 'RPI', 'EP', 'CL_PGE', 'SECT', 'SECTEUR', 'ACCES_AIRE_PRATIQUE', 'ACCES_LIBRE', 'ACCES_SANITAIRE', 'ACCES_VESTIAIRE', 'CAPACITE_D_ACCUEIL', 'PRES_DOUCHE', 'PRES_SANITAIRE', 'SAISONNIER', 'COUVERT', 'ECLAIRE', 'CATEGORIE', 'MULTIPLEXE', 'STRUCTURE_EXERCICE', 'SPECIALITE', 'ACCUEIL', 'ITINERANCE', 'MODE_GESTION', 'SSTYPHEB', 'TYPE', 'TYPERESTO', 'IMPLANTATION_STATION', 'GPL', 'CAPACITE', 'INDIC_CAPA', 'NBEQUIDENT', 'INDIC_NBEQUIDENT', 'NBSALLES', 'INDIC_NBSALLES', 'NBLIEUX', 'INDIC_NBLIEUX', 'NB_PDC', 'INDIC_NB_PDC', 'NB_PDC_PA', 'INDIC_NB_PDC_PA', 'NB_PDC_ACCELEREE', 'INDIC_NB_PDC_ACCELEREE', 'NB_PDC_LENTE', 'INDIC_NB_PDC_LENTE', 'NB_PDC_RAPIDE', 'INDIC_NB_PDC_RAPIDE', 'NB_PDC_ULTRARAPIDE', 'INDIC_NB_PDC_ULTRARAPIDE', 'NB_JOURS_OUVERT', 'INDIC_NB_JOURS_OUVE

In [7]:
#Identifier le fichier source

def empreinte_sha256(fichier):
    calcul = hashlib.sha256()

    with open(fichier, "rb") as flux:
        for bloc in iter(
            lambda: flux.read(1024 * 1024),
            b"",
        ):
            calcul.update(bloc)

    return calcul.hexdigest()


EMPREINTE_BPE = empreinte_sha256(FICHIER_BPE)

DATE_TRAITEMENT = datetime.now(
    timezone.utc
).isoformat(timespec="seconds")

print("SHA256 :", EMPREINTE_BPE)
print("Traitement UTC :", DATE_TRAITEMENT)

SHA256 : 47d4f3a1fa8e37c50d2821a70c630b77f0ea9350a947f5f269a1c90225dcbe48
Traitement UTC : 2026-09-23T10:55:55+00:00


In [8]:
#Extraire les restaurants IDF par blocs

DEPARTEMENTS_IDF = {
    "75", "77", "78", "91",
    "92", "93", "94", "95",
}

blocs_restaurants = []
nombre_lignes_lues = 0
nombre_restaurants_extraits = 0

with pd.read_csv(
    FICHIER_BPE,
    sep=SEPARATEUR_BPE,
    encoding=ENCODAGE_BPE,
    dtype="string",
    keep_default_na=False,
    chunksize=20_000,
) as lecteur:

    for numero_bloc, bloc in enumerate(lecteur, start=1):
        bloc = bloc.rename(columns=renommage_bpe)

        # Numéro de l'enregistrement, hors en-tête.
        bloc["RANG_SOURCE"] = np.arange(
            nombre_lignes_lues + 1,
            nombre_lignes_lues + len(bloc) + 1,
        )

        nombre_lignes_lues += len(bloc)

        code = normaliser_code_commune(bloc["DEPCOM"])

        type_equipement = (
            nettoyer_texte(bloc["TYPEQU"])
            .str.upper()
        )

        masque = (
            type_equipement.eq("A504").fillna(False)
            & code.str[:2].isin(DEPARTEMENTS_IDF)
        )

        extrait = bloc.loc[masque].copy()

        if not extrait.empty:
            blocs_restaurants.append(extrait)
            nombre_restaurants_extraits += len(extrait)

        if numero_bloc % 25 == 0:
            print(
                f"{nombre_lignes_lues:,} lignes lues ; "
                f"{nombre_restaurants_extraits:,} restaurants retenus"
            )

if not blocs_restaurants:
    raise ValueError(
        "Aucun restaurant A504 d'Île-de-France trouvé."
    )

restaurants_source = pd.concat(
    blocs_restaurants,
    ignore_index=True,
)

del blocs_restaurants

millesimes = set(
    nettoyer_texte(restaurants_source["AN"])
    .dropna()
)

if millesimes != {"2025"}:
    raise ValueError(
        f"Millésimes inattendus : {millesimes}"
    )

if nettoyer_texte(restaurants_source["AN"]).isna().any():
    raise ValueError("Le millésime manque pour certains restaurants.")

print("Restaurants extraits :", len(restaurants_source))
display(restaurants_source.head())

500,000 lignes lues ; 41,608 restaurants retenus
1,000,000 lignes lues ; 49,169 restaurants retenus
1,500,000 lignes lues ; 49,169 restaurants retenus
2,000,000 lignes lues ; 49,169 restaurants retenus
2,500,000 lignes lues ; 49,169 restaurants retenus
Restaurants extraits : 49169


,AN,APET,NOMRS,CNOMRS,NUMVOIE,INDREP,TYPVOIE,LIBVOIE,CADR,CODPOS,DEPCOM,DEP,REG,LIBCOM,DOM,SDOM,TYPEQU,SIRET,STATUT_DIFFUSION,CANTINE,INTERNAT,RPI,EP,CL_PGE,SECT,SECTEUR,ACCES_AIRE_PRATIQUE,ACCES_LIBRE,ACCES_SANITAIRE,ACCES_VESTIAIRE,CAPACITE_D_ACCUEIL,PRES_DOUCHE,PRES_SANITAIRE,SAISONNIER,COUVERT,ECLAIRE,CATEGORIE,MULTIPLEXE,STRUCTURE_EXERCICE,SPECIALITE,ACCUEIL,ITINERANCE,MODE_GESTION,SSTYPHEB,TYPE,TYPERESTO,IMPLANTATION_STATION,GPL,CAPACITE,INDIC_CAPA,NBEQUIDENT,INDIC_NBEQUIDENT,NBSALLES,INDIC_NBSALLES,NBLIEUX,INDIC_NBLIEUX,NB_PDC,INDIC_NB_PDC,NB_PDC_PA,INDIC_NB_PDC_PA,NB_PDC_ACCELEREE,INDIC_NB_PDC_ACCELEREE,NB_PDC_LENTE,INDIC_NB_PDC_LENTE,NB_PDC_RAPIDE,INDIC_NB_PDC_RAPIDE,NB_PDC_ULTRARAPIDE,INDIC_NB_PDC_ULTRARAPIDE,NB_JOURS_OUVERT,INDIC_NB_JOURS_OUVERT,LAMBERT_X,LAMBERT_Y,LONGITUDE,LATITUDE,QUALITE_XY,EPSG,QUALITE_GEOLOC,TR_DIST_PRECISION,DCIRIS,QUALI_IRIS,IRISEE,QP2024,QUALI_QP2024,QP2015,QUALI_QP2015,QVA,QUALI_QVA,ZUS,QUALI_ZUS,EPCI,UU2020,BV2022,AAV2020,DENS3,DENS7,RANG_SOURCE
0,2025,5610C,BOULANGERIES PAUL,,25,,AV,DE L OPERA,,75001,75101,75,11,PARIS 1ER ARRONDISSEMENT,A,A5,A504,40305211101212,O,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,5610C,_Z,_Z,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,651126,6863172,2.33377,48.86672,B,2154,11,< 100,751010301,1,1,CSZ,_Z,CSZ,_Z,CSZ,_Z,CSZ,_Z,200054781,00851,75056,001,1,1,93381
1,2025,5610C,CAFE SIRENE FRANCE,STARBUCKS COFFEE,26,,AV,DE L OPERA,,75001,75101,75,11,PARIS 1ER ARRONDISSEMENT,A,A5,A504,44533010300026,O,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,5610C,_Z,_Z,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,651148,6863212,2.33407,48.86708,B,2154,11,< 100,751010301,1,1,CSZ,_Z,CSZ,_Z,CSZ,_Z,CSZ,_Z,200054781,00851,75056,001,1,1,93382
2,2025,5610A,LE VICTORIA CAFE,,25,,AV,VICTORIA,,75001,75101,75,11,PARIS 1ER ARRONDISSEMENT,A,A5,A504,44444986200010,O,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,5610A,_Z,_Z,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,652032,6862171,2.34623,48.85779,B,2154,11,< 100,751010101,1,1,CSZ,_Z,CSZ,_Z,CSZ,_Z,CSZ,_Z,200054781,00851,75056,001,1,1,93383
3,2025,5610A,MANALOU,,7,,BD,DE LA MADELEINE,,75001,75101,75,11,PARIS 1ER ARRONDISSEMENT,A,A5,A504,47761841700018,O,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,5610A,_Z,_Z,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,650637,6863495,2.32706,48.86959,B,2154,11,< 100,751010402,1,1,CSZ,_Z,CSZ,_Z,CSZ,_Z,CSZ,_Z,200054781,00851,75056,001,1,1,93384
4,2025,5610C,CAFE SIRENE FRANCE,STARBUCKS COFFEE,11,,BD,DE SEBASTOPOL,,75001,75101,75,11,PARIS 1ER ARRONDISSEMENT,A,A5,A504,44533010300224,O,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,_Z,5610C,_Z,_Z,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,,0,652177,6862293,2.34819,48.8589,B,2154,11,< 100,751010201,1,1,CSZ,_Z,CSZ,_Z,CSZ,_Z,CSZ,_Z,200054781,00851,75056,001,1,1,93385


In [9]:
#Sauvegarder l'extraction intermédiaire

enregistrer_csv(
    restaurants_source,
    DOSSIER_INTERIM / "restaurants_extraits_bpe2025.csv",
)

restaurants = restaurants_source.copy()

colonnes_bpe_source = [
    colonne
    for colonne in restaurants_source.columns
    if colonne != "RANG_SOURCE"
]

colonnes_optionnelles = [
    "SIRET",
    "NOMRS",
    "CNOMRS",
    "NUMVOIE",
    "INDREP",
    "TYPVOIE",
    "LIBVOIE",
    "CADR",
    "CODPOS",
    "LIBCOM",
    "APET",
    "LATITUDE",
    "LONGITUDE",
    "DCIRIS",
    "QUALI_IRIS",
    "QUALITE_XY",
    "QUALITE_GEOLOC",
    "TR_DIST_PRECISION",
    "STATUT_DIFFUSION",
]

for colonne in colonnes_optionnelles:
    if colonne not in restaurants.columns:
        restaurants[colonne] = pd.Series(
            pd.NA,
            index=restaurants.index,
            dtype="string",
        )
        print("Colonne non disponible :", colonne)

restaurants_extraits_bpe2025.csv : 49,169 lignes


In [10]:
#Harmoniser les codes communaux

restaurants["CODGEO_SOURCE"] = normaliser_code_commune(
    restaurants["DEPCOM"]
)

restaurants["CODGEO"] = restaurants["CODGEO_SOURCE"]

codes_paris = {
    f"751{numero:02d}"
    for numero in range(1, 21)
}

if (
    "75056" in codes_profil
    and codes_profil.intersection(codes_paris)
):
    raise ValueError(
        "Le profil mélange Paris et ses arrondissements."
    )

if "75056" in codes_profil:
    masque_paris = restaurants["CODGEO"].isin(codes_paris)
    restaurants.loc[masque_paris, "CODGEO"] = "75056"

if "93066" in codes_profil and "93059" not in codes_profil:
    restaurants.loc[
        restaurants["CODGEO"].eq("93059"),
        "CODGEO",
    ] = "93066"

restaurants["CODE_COMMUNE_MODIFIE"] = (
    restaurants["CODGEO"] != restaurants["CODGEO_SOURCE"]
)

restaurants["COMMUNE_DANS_PROFIL"] = (
    restaurants["CODGEO"].isin(codes_profil)
)

diagnostic_communes = restaurants.loc[
    ~restaurants["COMMUNE_DANS_PROFIL"],
    ["RANG_SOURCE", "CODGEO_SOURCE", "CODGEO", "LIBCOM"],
].copy()

enregistrer_csv(
    diagnostic_communes,
    DOSSIER_INTERIM / "codes_non_apparies.csv",
)

print(
    "Observations hors profil :",
    len(diagnostic_communes),
)

codes_non_apparies.csv : 0 lignes
Observations hors profil : 0


In [11]:
#Contrôler les SIRET

siret_nettoye = (
    nettoyer_texte(restaurants["SIRET"])
    .str.replace(r"\s+", "", regex=True)
    .str.replace(r"\.0$", "", regex=True)
)

restaurants["SIRET_FORMAT_OK"] = (
    siret_nettoye.str.fullmatch(r"\d{14}", na=False)
    & siret_nettoye.ne("00000000000000").fillna(False)
)

restaurants["SIRET_NETTOYE"] = siret_nettoye.where(
    restaurants["SIRET_FORMAT_OK"]
)

print(
    "SIRET au format attendu :",
    restaurants["SIRET_FORMAT_OK"].sum(),
)

print(
    "SIRET absent ou format incorrect :",
    (~restaurants["SIRET_FORMAT_OK"]).sum(),
)

SIRET au format attendu : 49169
SIRET absent ou format incorrect : 0


In [12]:
#Traiter les répétitions certaines

masque_repetition = (
    restaurants["SIRET_FORMAT_OK"]
    & restaurants.duplicated(
        subset=colonnes_bpe_source,
        keep="first",
    )
)

repetitions_retirees = restaurants.loc[
    masque_repetition
].copy()

enregistrer_csv(
    repetitions_retirees,
    DOSSIER_INTERIM / "repetitions_identiques_retirees.csv",
)

restaurants = (
    restaurants.loc[~masque_repetition]
    .copy()
    .reset_index(drop=True)
)

restaurants["SIRET_REPETE"] = (
    restaurants["SIRET_NETTOYE"].notna()
    & restaurants["SIRET_NETTOYE"].duplicated(keep=False)
)

conflits_siret = restaurants.loc[
    restaurants["SIRET_REPETE"]
].sort_values("SIRET_NETTOYE")

enregistrer_csv(
    conflits_siret,
    DOSSIER_INTERIM / "siret_repetes_a_examiner.csv",
)

print("Répétitions identiques retirées :", len(repetitions_retirees))
print("Observations avec SIRET répété :", len(conflits_siret))

repetitions_identiques_retirees.csv : 0 lignes
siret_repetes_a_examiner.csv : 0 lignes
Répétitions identiques retirées : 0
Observations avec SIRET répété : 0


In [13]:
#Créer un identifiant technique traçable

restaurants["ID_OBSERVATION"] = (
    "BPE2025_"
    + EMPREINTE_BPE[:16]
    + "_"
    + restaurants["RANG_SOURCE"].astype("string")
)

assert restaurants["ID_OBSERVATION"].is_unique

restaurants["SOURCE_DONNEES"] = "INSEE_BPE_2025"
restaurants["FICHIER_SOURCE"] = FICHIER_BPE.name
restaurants["SHA256_SOURCE"] = EMPREINTE_BPE
restaurants["DATE_TRAITEMENT_UTC"] = DATE_TRAITEMENT

restaurants["STATUT_OUVERTURE_ACTUELLE"] = "NON_VERIFIE"

display(
    restaurants[
        [
            "ID_OBSERVATION",
            "SIRET",
            "SIRET_NETTOYE",
            "SIRET_FORMAT_OK",
        ]
    ].head()
)

,ID_OBSERVATION,SIRET,SIRET_NETTOYE,SIRET_FORMAT_OK
0,BPE2025_47d4f3a1fa8e37c5_93381,40305211101212,40305211101212,True
1,BPE2025_47d4f3a1fa8e37c5_93382,44533010300026,44533010300026,True
2,BPE2025_47d4f3a1fa8e37c5_93383,44444986200010,44444986200010,True
3,BPE2025_47d4f3a1fa8e37c5_93384,47761841700018,47761841700018,True
4,BPE2025_47d4f3a1fa8e37c5_93385,44533010300224,44533010300224,True


In [15]:
#Classer les restaurants

correspondance_categories = {
    "5610A": "TRADITIONNELLE",
    "5610B": "CAFETERIA_LIBRE_SERVICE",
    "5610C": "RAPIDE",
}

restaurants["TYPERESTO_NETTOYE"] = (
    nettoyer_texte(restaurants["TYPERESTO"])
    .str.upper()
    .str.replace(".", "", regex=False)
    .str.replace(" ", "", regex=False)
)

restaurants["CATEGORIE_RESTAURATION"] = (
    restaurants["TYPERESTO_NETTOYE"]
    .map(correspondance_categories)
    .fillna("TYPE_INCONNU")
)

restaurants["TYPE_RESTAURATION_CONNU"] = (
    restaurants["CATEGORIE_RESTAURATION"] != "TYPE_INCONNU"
)

display(
    restaurants[
        [
            "TYPERESTO",
            "TYPERESTO_NETTOYE",
            "CATEGORIE_RESTAURATION",
        ]
    ].value_counts(dropna=False)
)

enregistrer_csv(
    restaurants.loc[
        ~restaurants["TYPE_RESTAURATION_CONNU"]
    ],
    DOSSIER_INTERIM / "types_restaurants_inconnus.csv",
)

TYPERESTO  TYPERESTO_NETTOYE  CATEGORIE_RESTAURATION 
5610A      5610A              TRADITIONNELLE             24764
5610C      5610C              RAPIDE                     24319
5610B      5610B              CAFETERIA_LIBRE_SERVICE       86
Name: count, dtype: int64

types_restaurants_inconnus.csv : 0 lignes


In [16]:
#Préparer les noms et adresses

restaurants["NOM_SOURCE"] = nettoyer_texte(
    restaurants["NOMRS"]
)

restaurants["COMPLEMENT_NOM_SOURCE"] = nettoyer_texte(
    restaurants["CNOMRS"]
)

restaurants["NOM_AFFICHAGE"] = (
    restaurants["NOM_SOURCE"]
    .fillna(restaurants["COMPLEMENT_NOM_SOURCE"])
)

colonnes_adresse = [
    "NUMVOIE",
    "INDREP",
    "TYPVOIE",
    "LIBVOIE",
    "CADR",
]

parties_adresse = pd.DataFrame({
    colonne: nettoyer_texte(restaurants[colonne])
    for colonne in colonnes_adresse
})

restaurants["ADRESSE_SOURCE_RECOMPOSEE"] = (
    parties_adresse.fillna("")
    .agg(" ".join, axis=1)
    .str.replace(r"\s+", " ", regex=True)
    .str.strip()
    .replace("", pd.NA)
)

restaurants["CODE_POSTAL_NETTOYE"] = normaliser_code_commune(
    restaurants["CODPOS"]
)

restaurants["NOM_DISPONIBLE"] = (
    restaurants["NOM_AFFICHAGE"].notna()
)

restaurants["VOIE_DISPONIBLE"] = (
    nettoyer_texte(restaurants["LIBVOIE"]).notna()
)

display(
    restaurants[
        [
            "NOM_AFFICHAGE",
            "ADRESSE_SOURCE_RECOMPOSEE",
            "CODE_POSTAL_NETTOYE",
            "CODGEO",
        ]
    ].head(10)
)

,NOM_AFFICHAGE,ADRESSE_SOURCE_RECOMPOSEE,CODE_POSTAL_NETTOYE,CODGEO
0,BOULANGERIES PAUL,25 AV DE L OPERA,75001,75056
1,CAFE SIRENE FRANCE,26 AV DE L OPERA,75001,75056
2,LE VICTORIA CAFE,25 AV VICTORIA,75001,75056
3,MANALOU,7 BD DE LA MADELEINE,75001,75056
4,CAFE SIRENE FRANCE,11 BD DE SEBASTOPOL,75001,75056
5,AUX FRONTIERES DU DESERT,55 BD DE SEBASTOPOL,75001,75056
6,POMONE,JARD DES TUILERIES DOMAINE NAT DU LOUVRE TU,75001,75056
7,ORFINE,15 PL DAUPHINE,75001,75056
8,LE CAVEAU DU PALAIS,19 PL DAUPHINE,75001,75056
9,BEYRIS,26 PL DAUPHINE,75001,75056


In [17]:
#Signaler les ressemblances nom-adresse

restaurants["NOM_COMPARAISON"] = texte_comparable(
    restaurants["NOM_AFFICHAGE"]
)

restaurants["ADRESSE_COMPARAISON"] = texte_comparable(
    restaurants["ADRESSE_SOURCE_RECOMPOSEE"]
)

cle_comparaison = [
    "CODGEO",
    "NOM_COMPARAISON",
    "ADRESSE_COMPARAISON",
]

informations_suffisantes = (
    restaurants["NOM_COMPARAISON"].fillna("").ne("")
    & restaurants["ADRESSE_COMPARAISON"].fillna("").ne("")
    & restaurants["VOIE_DISPONIBLE"]
)

restaurants["RESSEMBLANCE_NOM_ADRESSE"] = (
    informations_suffisantes
    & restaurants.duplicated(
        subset=cle_comparaison,
        keep=False,
    )
)

ressemblances = restaurants.loc[
    restaurants["RESSEMBLANCE_NOM_ADRESSE"]
].sort_values(cle_comparaison)

enregistrer_csv(
    ressemblances,
    DOSSIER_INTERIM / "ressemblances_nom_adresse.csv",
)

print("Observations à comparer :", len(ressemblances))

ressemblances_nom_adresse.csv : 76 lignes
Observations à comparer : 76


In [18]:
#Examiner les coordonnées

restaurants["LATITUDE_NUM"] = convertir_nombre(
    restaurants["LATITUDE"]
)

restaurants["LONGITUDE_NUM"] = convertir_nombre(
    restaurants["LONGITUDE"]
)

coordonnees_presentes = (
    restaurants["LATITUDE_NUM"].notna()
    & restaurants["LONGITUDE_NUM"].notna()
)

# Rectangle large de contrôle autour de l'Île-de-France.
# Ce n'est pas un contrôle d'appartenance aux limites communales.
coordonnees_plausibles = (
    restaurants["LATITUDE_NUM"].between(48.0, 49.3)
    & restaurants["LONGITUDE_NUM"].between(1.4, 3.7)
).fillna(False)

restaurants["COORDONNEES_PLAUSIBLES_IDF"] = (
    coordonnees_plausibles
)

restaurants["CONTROLE_COORDONNEES"] = "ABSENTES_OU_NON_NUMERIQUES"

restaurants.loc[
    coordonnees_presentes,
    "CONTROLE_COORDONNEES",
] = "HORS_RECTANGLE_CONTROLE"

restaurants.loc[
    coordonnees_plausibles,
    "CONTROLE_COORDONNEES",
] = "PLAUSIBLES_A_VERIFIER"

display(
    restaurants["CONTROLE_COORDONNEES"]
    .value_counts(dropna=False)
)

enregistrer_csv(
    restaurants.loc[~coordonnees_plausibles],
    DOSSIER_INTERIM / "coordonnees_a_examiner.csv",
)

CONTROLE_COORDONNEES
PLAUSIBLES_A_VERIFIER         47856
ABSENTES_OU_NON_NUMERIQUES     1313
Name: count, dtype: int64

coordonnees_a_examiner.csv : 1,313 lignes


In [19]:
#Conserve les informations de qualité et d'IRIS

restaurants["CODE_IRIS_SOURCE"] = nettoyer_texte(
    restaurants["DCIRIS"]
)

restaurants["IRIS_FORMAT_OK"] = (
    restaurants["CODE_IRIS_SOURCE"]
    .str.fullmatch(r"\d{9}", na=False)
)

colonnes_qualite = [
    "QUALITE_XY",
    "QUALITE_GEOLOC",
    "TR_DIST_PRECISION",
    "QUALI_IRIS",
    "STATUT_DIFFUSION",
]

for colonne in colonnes_qualite:
    print("\n", colonne)

    display(
        restaurants[colonne]
        .value_counts(dropna=False)
        .head(15)
    )

print(
    "Codes IRIS au format attendu :",
    restaurants["IRIS_FORMAT_OK"].sum(),
)


 QUALITE_XY


QUALITE_XY
B     46709
_U     1313
M       810
A       337
Name: count, dtype: Int64


 QUALITE_GEOLOC


QUALITE_GEOLOC
11    46709
_U     1313
12      641
33      235
21      160
22      111
Name: count, dtype: Int64


 TR_DIST_PRECISION


...


 QUALI_IRIS


QUALI_IRIS
1     44617
_Z     2767
3      1385
2       400
Name: count, dtype: Int64


 STATUT_DIFFUSION


STATUT_DIFFUSION
O    47856
P     1313
Name: count, dtype: Int64

Codes IRIS au format attendu : 47948


In [20]:
#Produire un bilan de qualité

indicateurs_qualite = {
    "SIRET_FORMAT_OK": restaurants["SIRET_FORMAT_OK"],
    "NOM_DISPONIBLE": restaurants["NOM_DISPONIBLE"],
    "VOIE_DISPONIBLE": restaurants["VOIE_DISPONIBLE"],
    "TYPE_RESTAURATION_CONNU": restaurants["TYPE_RESTAURATION_CONNU"],
    "COORDONNEES_PLAUSIBLES_IDF": restaurants["COORDONNEES_PLAUSIBLES_IDF"],
    "IRIS_FORMAT_OK": restaurants["IRIS_FORMAT_OK"],
    "COMMUNE_DANS_PROFIL": restaurants["COMMUNE_DANS_PROFIL"],
    "SIRET_REPETE": restaurants["SIRET_REPETE"],
    "RESSEMBLANCE_NOM_ADRESSE": restaurants["RESSEMBLANCE_NOM_ADRESSE"],
}

bilan_qualite = pd.DataFrame([
    {
        "INDICATEUR": nom,
        "NB_OBSERVATIONS": int(masque.sum()),
        "PART_PCT": round(
            100 * masque.mean(),
            2,
        ),
    }
    for nom, masque in indicateurs_qualite.items()
])

display(bilan_qualite)

enregistrer_csv(
    bilan_qualite,
    DOSSIER_INTERIM / "bilan_qualite.csv",
)

,INDICATEUR,NB_OBSERVATIONS,PART_PCT
0,SIRET_FORMAT_OK,49169,100.00
1,NOM_DISPONIBLE,47890,97.40
2,VOIE_DISPONIBLE,47856,97.33
3,TYPE_RESTAURATION_CONNU,49169,100.00
4,COORDONNEES_PLAUSIBLES_IDF,47856,97.33
5,IRIS_FORMAT_OK,47948,97.52
6,COMMUNE_DANS_PROFIL,49169,100.00
7,SIRET_REPETE,0,0.00
8,RESSEMBLANCE_NOM_ADRESSE,76,0.15


bilan_qualite.csv : 9 lignes


In [21]:
#Recalculer les comptages communaux

correspondance_comptages = {
    "TRADITIONNELLE": "NB_RESTAURATION_TRADITIONNELLE",
    "CAFETERIA_LIBRE_SERVICE": "NB_CAFETERIAS_LIBRE_SERVICE",
    "RAPIDE": "NB_RESTAURATION_RAPIDE",
    "TYPE_INCONNU": "NB_RESTAURATION_TYPE_INCONNU",
}

table_categories = pd.crosstab(
    restaurants["CODGEO"],
    restaurants["CATEGORIE_RESTAURATION"],
)

table_categories = table_categories.reindex(
    columns=list(correspondance_comptages),
    fill_value=0,
)

comptages = (
    table_categories
    .rename(columns=correspondance_comptages)
    .reset_index()
)

comptages.columns.name = None

colonnes_categories = list(
    correspondance_comptages.values()
)

comptages["NB_RESTAURANTS_TOTAL"] = (
    comptages[colonnes_categories].sum(axis=1)
)

colonnes_comptages = [
    "NB_RESTAURANTS_TOTAL",
    *colonnes_categories,
]

comptages[colonnes_comptages] = (
    comptages[colonnes_comptages].astype(int)
)

qualite_communes = (
    restaurants.groupby("CODGEO", as_index=False)
    .agg(
        NB_OBS_COORDONNEES_PLAUSIBLES=(
            "COORDONNEES_PLAUSIBLES_IDF",
            "sum",
        ),
        NB_OBS_SIRET_FORMAT_OK=(
            "SIRET_FORMAT_OK",
            "sum",
        ),
        NB_OBS_SIRET_REPETE=(
            "SIRET_REPETE",
            "sum",
        ),
    )
)

comptages = comptages.merge(
    qualite_communes,
    on="CODGEO",
    how="left",
    validate="one_to_one",
)

display(comptages.head())

,CODGEO,NB_RESTAURATION_TRADITIONNELLE,NB_CAFETERIAS_LIBRE_SERVICE,NB_RESTAURATION_RAPIDE,NB_RESTAURATION_TYPE_INCONNU,NB_RESTAURANTS_TOTAL,NB_OBS_COORDONNEES_PLAUSIBLES,NB_OBS_SIRET_FORMAT_OK,NB_OBS_SIRET_REPETE
0,75056,12553,29,8329,0,20911,20583,20911,0
1,77001,1,0,0,0,1,1,1,0
2,77002,0,0,2,0,2,2,2,0
3,77003,1,0,0,0,1,1,1,0
4,77005,4,0,2,0,6,6,6,0


In [22]:
#Comparer avec le profil J9

colonnes_absentes_profil = [
    colonne
    for colonne in colonnes_comptages
    if colonne not in profil_j9.columns
]

if colonnes_absentes_profil:
    raise ValueError(
        "Comptages J6 absents du profil J9 : "
        f"{colonnes_absentes_profil}"
    )

avant = profil_j9[
    ["CODGEO", *colonnes_comptages]
].copy()

for colonne in colonnes_comptages:
    avant[colonne] = convertir_nombre(avant[colonne])

    if avant[colonne].isna().any():
        raise ValueError(
            f"Le profil J9 contient des valeurs manquantes "
            f"ou non numériques dans {colonne}."
        )

    if (
        avant[colonne].lt(0).any()
        or avant[colonne].mod(1).ne(0).any()
    ):
        raise ValueError(
            f"Comptages invalides dans {colonne}."
        )

    avant[colonne] = avant[colonne].astype(int)

comparaison = avant.merge(
    comptages[["CODGEO", *colonnes_comptages]],
    on="CODGEO",
    how="outer",
    suffixes=("_J9", "_J9B"),
    indicator=True,
    validate="one_to_one",
)

# Pour une commune du profil sans observation dans l'extraction,
# le nombre d'observations BPE retenues est zéro.
dans_profil = comparaison["_merge"].ne("right_only")

for colonne in colonnes_comptages:
    colonne_nouvelle = f"{colonne}_J9B"

    comparaison.loc[dans_profil, colonne_nouvelle] = (
        comparaison.loc[dans_profil, colonne_nouvelle]
        .fillna(0)
    )

    comparaison[f"ECART_{colonne}"] = (
        comparaison[colonne_nouvelle]
        - comparaison[f"{colonne}_J9"]
    )

colonnes_ecarts = [
    f"ECART_{colonne}"
    for colonne in colonnes_comptages
]

comparaison["ECART_OU_HORS_PROFIL"] = (
    comparaison["_merge"].eq("right_only")
    | comparaison[colonnes_ecarts]
    .fillna(0)
    .ne(0)
    .any(axis=1)
)

ecarts = comparaison.loc[
    comparaison["ECART_OU_HORS_PROFIL"]
].copy()

display(ecarts.head(30))

enregistrer_csv(
    comparaison,
    DOSSIER_INTERIM / "comparaison_comptages_j9_j9b.csv",
)

print("Communes avec écart ou hors profil :", len(ecarts))

,CODGEO,NB_RESTAURANTS_TOTAL_J9,NB_RESTAURATION_TRADITIONNELLE_J9,NB_CAFETERIAS_LIBRE_SERVICE_J9,NB_RESTAURATION_RAPIDE_J9,NB_RESTAURATION_TYPE_INCONNU_J9,NB_RESTAURANTS_TOTAL_J9B,NB_RESTAURATION_TRADITIONNELLE_J9B,NB_CAFETERIAS_LIBRE_SERVICE_J9B,NB_RESTAURATION_RAPIDE_J9B,NB_RESTAURATION_TYPE_INCONNU_J9B,_merge,ECART_NB_RESTAURANTS_TOTAL,ECART_NB_RESTAURATION_TRADITIONNELLE,ECART_NB_CAFETERIAS_LIBRE_SERVICE,ECART_NB_RESTAURATION_RAPIDE,ECART_NB_RESTAURATION_TYPE_INCONNU,ECART_OU_HORS_PROFIL


comparaison_comptages_j9_j9b.csv : 1,266 lignes
Communes avec écart ou hors profil : 0


In [23]:
#Définir l'état de contrôle

points_a_resoudre = []

if restaurants["SIRET_REPETE"].any():
    points_a_resoudre.append(
        "SIRET répétés restant à examiner"
    )

if not diagnostic_communes.empty:
    points_a_resoudre.append(
        "Observations rattachées à des communes absentes du profil"
    )

if not ecarts.empty:
    points_a_resoudre.append(
        "Comptages différents de ceux du profil J9"
    )

ETAT_CONTROLE = (
    "A_RECONCILIER"
    if points_a_resoudre
    else "COHERENT_AVEC_J9"
)

restaurants["ETAT_CONTROLE_J9B"] = ETAT_CONTROLE
comptages["ETAT_CONTROLE_J9B"] = ETAT_CONTROLE

print("État :", ETAT_CONTROLE)

for point in points_a_resoudre:
    print("-", point)

print(
    "Cet état porte sur les contrôles de cohérence ; "
    "il ne confirme pas l'ouverture actuelle des restaurants."
)

État : COHERENT_AVEC_J9
Cet état porte sur les contrôles de cohérence ; il ne confirme pas l'ouverture actuelle des restaurants.


In [24]:
#Enregistrer la base détaillée

colonnes_prioritaires = [
    "ID_OBSERVATION",
    "SIRET_NETTOYE",
    "SIRET_FORMAT_OK",
    "SIRET_REPETE",
    "CODGEO",
    "CODGEO_SOURCE",
    "COMMUNE_DANS_PROFIL",
    "NOM_AFFICHAGE",
    "ADRESSE_SOURCE_RECOMPOSEE",
    "CODE_POSTAL_NETTOYE",
    "CATEGORIE_RESTAURATION",
    "LATITUDE_NUM",
    "LONGITUDE_NUM",
    "CONTROLE_COORDONNEES",
    "QUALITE_XY",
    "QUALITE_GEOLOC",
    "TR_DIST_PRECISION",
    "CODE_IRIS_SOURCE",
    "QUALI_IRIS",
    "STATUT_DIFFUSION",
    "STATUT_OUVERTURE_ACTUELLE",
    "ETAT_CONTROLE_J9B",
    "SOURCE_DONNEES",
    "SHA256_SOURCE",
    "DATE_TRAITEMENT_UTC",
]

autres_colonnes = [
    colonne
    for colonne in restaurants.columns
    if colonne not in colonnes_prioritaires
]

restaurants_detail = restaurants[
    colonnes_prioritaires + autres_colonnes
].copy()

assert restaurants_detail["ID_OBSERVATION"].is_unique

assert (
    len(restaurants_source)
    == len(restaurants_detail) + len(repetitions_retirees)
)

assert comptages["CODGEO"].is_unique

assert (
    comptages["NB_RESTAURANTS_TOTAL"]
    == comptages[colonnes_categories].sum(axis=1)
).all()

assert (
    comptages["NB_RESTAURANTS_TOTAL"].sum()
    == len(restaurants_detail)
)

enregistrer_csv(
    restaurants_detail,
    FICHIER_RESTAURANTS_DETAIL,
)

enregistrer_csv(
    comptages,
    DOSSIER_INTERIM / "comptages_restaurants_communes.csv",
)

restaurants_idf_detail_j9b.csv : 49,169 lignes
comptages_restaurants_communes.csv : 932 lignes


In [25]:
#Préparer le futur enrichissement commercial

colonnes_collecte = [
    "ID_OBSERVATION",
    "NOM_COMMERCIAL_VERIFIE",
    "CUISINE",
    "CONCEPT",
    "ENSEIGNE",
    "APPARTENANCE_RESEAU",
    "PRIX_FORMULE_MIDI_EUR",
    "DATE_RELEVE_PRIX",
    "HORAIRES_OBSERVES",
    "SUR_PLACE",
    "A_EMPORTER",
    "LIVRAISON",
    "STATUT_OUVERTURE_OBSERVE",
    "DATE_VERIFICATION",
    "SOURCE_VERIFICATION",
    "COMMENTAIRE",
]

FICHIER_MODELE_COLLECTE = (
    DOSSIER_INTERIM / "modele_collecte_commerciale.csv"
)

# Préserver les saisies si le fichier a déjà été utilisé.
if not FICHIER_MODELE_COLLECTE.exists():
    enregistrer_csv(
        pd.DataFrame(columns=colonnes_collecte),
        FICHIER_MODELE_COLLECTE,
    )
else:
    print("Modèle déjà présent : conservé.")

modele_collecte_commerciale.csv : 0 lignes


In [26]:
#Enregistrer la tracabilité du traitement

metadonnees = {
    "notebook": "09b_restaurants_detail_idf.ipynb",
    "source": "Insee - BPE 2025",
    "url_source": "https://www.insee.fr/fr/statistiques/8217525",
    "fichier_source": str(FICHIER_BPE.relative_to(RACINE)),
    "sha256_source": EMPREINTE_BPE,
    "millesime": 2025,
    "date_traitement_utc": DATE_TRAITEMENT,
    "date_telechargement": None,
    "perimetre": "TYPEQU A504, départements d'Île-de-France",
    "nb_observations_extraites": int(len(restaurants_source)),
    "nb_repetitions_retirees": int(len(repetitions_retirees)),
    "nb_observations_conservees": int(len(restaurants_detail)),
    "etat_controle": ETAT_CONTROLE,
    "points_a_resoudre": points_a_resoudre,
    "limites": [
        "Ouverture actuelle non vérifiée",
        "Format SIRET contrôlé sans validation administrative",
        "Coordonnées plausibles non validées sur les contours",
        "Offre commerciale détaillée à compléter au J13b",
    ],
}

FICHIER_METADATA = DOSSIER_INTERIM / "metadata_j9b.json"

FICHIER_METADATA.write_text(
    json.dumps(
        metadonnees,
        ensure_ascii=False,
        indent=2,
    ),
    encoding="utf-8",
)

print("Traçabilité enregistrée :", FICHIER_METADATA.name)
print("Base détaillée :", FICHIER_RESTAURANTS_DETAIL.name)
print("État du contrôle :", ETAT_CONTROLE)

Traçabilité enregistrée : metadata_j9b.json
Base détaillée : restaurants_idf_detail_j9b.csv
État du contrôle : COHERENT_AVEC_J9
